# GNNs

### Architecture description
- SAGEConv 2 layer (better than GCNConv for such heterogeneous graph)
- DropEdge during training to avoid overfitting
- Symetric decoder : $|z_u * z_v | \sqcup |z_u - z_v| \sqcup | f_{struct}(u), f_{struct}(v) |$
- Structural features are computed before creating the PyG graph, in order to compute them only on the training mask and avoid information leakage.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.utils import dropout_edge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and split

In [ ]:
train_full = pd.read_csv("data/train.txt", sep=" ", header=None)
train_full.columns = ["u", "v", "label"]
train_full = train_full[train_full["u"] != train_full["v"]].reset_index(drop=True)  # Gets rid of self loop if any

test = pd.read_csv("data/test.txt", sep=" ", header=None)
test.columns = ["u", "v"]

node_info = pd.read_csv("data/node_information.csv", header=None)
node_info = node_info.rename(columns={0: "node"}).set_index("node")
features_df = node_info.copy() 

node_features_raw = {int(idx): row.values.astype(np.float32) for idx, row in features_df.iterrows()}

# Set of non-zero feature per nodes (will be used for calculating NFO)
feature_sets = {int(idx): set(np.where(row.values != 0)[0]) for idx, row in features_df.iterrows()}
feat_norms = {n: np.linalg.norm(f) + 1e-9 for n, f in node_features_raw.items()}
feature_dim = features_df.shape[1]

# Train/val split
train_df, val_df = train_test_split(train_full, test_size=0.2, stratify=train_full["label"], random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test)}")
print(f"Dim of node features : {feature_dim} dims")

## 2. Train masked graph

In [ ]:
G = nx.Graph()
G.add_edges_from(train_df[train_df["label"] == 1][["u", "v"]].values)

for u, v in pd.concat([train_full[["u","v"]], test[["u","v"]]]).values:
    u, v = int(u), int(v)
    if not G.has_node(u): G.add_node(u)
    if not G.has_node(v): G.add_node(v)

degree = dict(G.degree())
components = {n: c for c, comp in enumerate(nx.connected_components(G)) for n in comp}

print(f"Nodes : {G.number_of_nodes()} | Edges (in train mask) : {G.number_of_edges()}")

## 3. Nodes-pair features

- Symmetric normalized NFO

- #common_neighbors × cosine_similarity

In [ ]:
def cosine_sim(u, v):
    if u not in node_features_raw or v not in node_features_raw:
        return 0.0
    return float(np.dot(node_features_raw[u], node_features_raw[v]) / (feat_norms[u] * feat_norms[v]))


def nfo(u, v):
    """
    Neighbor Feature Overlap (u to v, non symetric), normalized with deg(v).
    """
    if not G.has_node(v) or not G.has_node(u):
        return 0.0
    neighbors_v = list(G.neighbors(v))
    if len(neighbors_v) == 0 or u not in feature_sets:
        return 0.0
    Fu    = feature_sets[u]
    count = sum(len(Fu & feature_sets[n]) for n in neighbors_v if n in feature_sets)
    return count / len(neighbors_v)


def pair_features(u, v):
    """
    Returns feature vector of pair (u, v).
    relevant features :
    - deg_u
    - deg_v
    - common_neighbors (CN)
    - adamic_adar
    - preferential_attachment
    - same_component
    - NFO_sym normalized
    - cn * cosine
    """
    deg_u = degree.get(u, 0)
    deg_v = degree.get(v, 0)

    if G.has_node(u) and G.has_node(v):
        nu = set(G.neighbors(u))
        nv = set(G.neighbors(v))
        inter = nu & nv
        cn = len(inter)
        aa = sum(1.0 / np.log(degree[w] + 1e-9) for w in inter if degree.get(w, 0) > 1)
        pa = deg_u * deg_v
        sc = float(components.get(u, -1) == components.get(v, -2))
    else:
        cn, aa, pa, sc = 0, 0.0, 0, 0.0

    nfo_sym = nfo(u, v) + nfo(v, u)
    cos = cosine_sim(u, v)
    cn_cosine = cn * cos

    return np.array([deg_u, deg_v, cn, aa, pa, sc, nfo_sym, cn_cosine], dtype=np.float32)


print("Nodes pair features computation")
train_struct = np.stack([pair_features(r.u, r.v) for r in train_df.itertuples()])
val_struct = np.stack([pair_features(r.u, r.v) for r in val_df.itertuples()])
test_struct = np.stack([pair_features(r.u, r.v) for r in test.itertuples()])

scaler = StandardScaler().fit(train_struct)
train_struct = scaler.transform(train_struct).astype(np.float32)
val_struct = scaler.transform(val_struct).astype(np.float32)
test_struct = scaler.transform(test_struct).astype(np.float32)
struct_dim = train_struct.shape[1]
print(f"Dim pair features : {struct_dim}")

## 4. PyG graph (node features with L2 normalization)

In [ ]:
all_nodes   = sorted(G.nodes())
node_to_idx = {n: i for i, n in enumerate(all_nodes)}
num_nodes   = len(all_nodes)

x_np = np.zeros((num_nodes, feature_dim), dtype=np.float32)
for node, idx in node_to_idx.items():
    if node in node_features_raw:
        x_np[idx] = node_features_raw[node]

# L2 Norm
norms = np.linalg.norm(x_np, axis=1, keepdims=True) + 1e-9
x_np = x_np / norms

x = torch.tensor(x_np, dtype=torch.float)
edge_list = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

data = Data(x=x, edge_index=edge_index, num_nodes=num_nodes)
print(data)

In [ ]:
def pairs_to_tensors(df, struct_arr, node_to_idx):
    mask  = df["u"].isin(node_to_idx) & df["v"].isin(node_to_idx)
    idx   = np.where(mask.values)[0]
    valid = df.iloc[idx]
    return (
        torch.tensor([node_to_idx[u] for u in valid["u"]], dtype=torch.long),
        torch.tensor([node_to_idx[v] for v in valid["v"]], dtype=torch.long),
        torch.tensor(struct_arr[idx], dtype=torch.float),
        torch.tensor(valid["label"].values, dtype=torch.float),
    )

train_u, train_v, train_s, train_y = pairs_to_tensors(train_df, train_struct, node_to_idx)
val_u, val_v, val_s, val_y = pairs_to_tensors(val_df, val_struct, node_to_idx)
print(f"#pairs = train : {len(train_y)} | val : {len(val_y)}")

## 5. Models

In [ ]:
class GNN_model(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.4):
        super().__init__()
        self.conv1 = SAGEConv(input_size, hidden_size)
        self.bn1 = torch.nn.BatchNorm1d(hidden_size)
        self.conv2 = SAGEConv(hidden_size, output_size)
        self.bn2 = torch.nn.BatchNorm1d(output_size)
        self.dropout = dropout

    def forward(self, x, edge_index, drop_edge_p=0.0):
        if drop_edge_p > 0:
            edge_index, _ = dropout_edge(edge_index, p=drop_edge_p, training=self.training)
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.bn2(self.conv2(x, edge_index))
        return x


class LinkPredictor(torch.nn.Module):
    """
    Symetric decoder
    """
    def __init__(self, emb_dim, struct_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        in_dim = emb_dim * 2 + struct_dim
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim),
            torch.nn.BatchNorm1d(hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim, hidden_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, z_u, z_v, s):
        return self.net(torch.cat([z_u * z_v, torch.abs(z_u - z_v), s], dim=-1)).squeeze(-1)

## 6. Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
data = data.to(device)
train_u, train_v, train_s, train_y = [t.to(device) for t in (train_u, train_v, train_s, train_y)]
val_u, val_v, val_s, val_y = [t.to(device) for t in (val_u, val_v, val_s, val_y)]

HIDDEN_SIZE = 256
EMB_SIZE = 128
DROPOUT_GNN = 0.4
DROPOUT_MLP = 0.3
DROP_EDGE_P = 0.4
LR = 1e-3
WEIGHT_DECAY = 5e-4
EPOCHS = 600
PATIENCE = 80

gnn = GNN_model(feature_dim, HIDDEN_SIZE, EMB_SIZE, DROPOUT_GNN).to(device)
predictor = LinkPredictor(EMB_SIZE, struct_dim, HIDDEN_SIZE, DROPOUT_MLP).to(device)
optimizer = torch.optim.Adam(list(gnn.parameters()) + list(predictor.parameters()),lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=30, verbose=True)
pos_weight = torch.tensor([(train_y == 0).sum() / (train_y == 1).sum()]).to(device)


def evaluate(z, u, v, s, y):
    with torch.no_grad():
        logits = predictor(z[u], z[v], s)
        loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
        auc = roc_auc_score(y.cpu().numpy(), torch.sigmoid(logits).cpu().numpy())
    return loss.item(), auc


best_val_auc = 0.0
best_state = None
epochs_no_imp = 0

for epoch in range(1, EPOCHS + 1):
    gnn.train(); predictor.train()
    optimizer.zero_grad()
    z = gnn(data.x, data.edge_index, drop_edge_p=DROP_EDGE_P)
    logits = predictor(z[train_u], z[train_v], train_s)
    loss = F.binary_cross_entropy_with_logits(logits, train_y, pos_weight=pos_weight)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        gnn.eval(); predictor.eval()
        z_e = gnn(data.x, data.edge_index, drop_edge_p=0.0)
        tr_loss, tr_auc = evaluate(z_e, train_u, train_v, train_s, train_y)
        va_loss, va_auc = evaluate(z_e, val_u, val_v, val_s, val_y)
        scheduler.step(va_auc)
        print(f"Epoch {epoch:>3} | Train loss {tr_loss:.4f} AUC {tr_auc:.4f} | Val loss {va_loss:.4f} AUC {va_auc:.4f}")
        if va_auc > best_val_auc:
            best_val_auc = va_auc
            epochs_no_imp = 0
            BEST_EPOCH = epoch
            best_state = {"gnn": {k: v.cpu().clone() for k, v in gnn.state_dict().items()}, "predictor": {k: v.cpu().clone() for k, v in predictor.state_dict().items()}}
        else:
            epochs_no_imp += 20
            if epochs_no_imp >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

print(f"\nBest validation AUC : {best_val_auc:.4f}")

## 7. Re-training on the full train set
This allowed to gain a few percentage point on Kaggle, as we know roughly at which epoch to stop the training with the previous validation, and we are able to leverage the full training set

In [ ]:
# Rebuild graph and recomputing features on the full training set
G_full = nx.Graph()
G_full.add_edges_from(train_full[train_full["label"] == 1][["u", "v"]].values)
for u, v in test[["u", "v"]].values:
    if not G_full.has_node(int(u)): G_full.add_node(int(u))
    if not G_full.has_node(int(v)): G_full.add_node(int(v))

degree_full = dict(G_full.degree())
components_full = {n: c for c, comp in enumerate(nx.connected_components(G_full)) for n in comp}


def nfo_full(u, v):
    if not G_full.has_node(v) or u not in feature_sets: return 0.0
    nb = list(G_full.neighbors(v))
    if not nb: return 0.0
    return sum(len(feature_sets[u] & feature_sets[n]) for n in nb if n in feature_sets) / len(nb)


def pair_features_full(u, v):
    deg_u = degree_full.get(u, 0); deg_v = degree_full.get(v, 0)
    if G_full.has_node(u) and G_full.has_node(v):
        inter = set(G_full.neighbors(u)) & set(G_full.neighbors(v))
        cn = len(inter)
        aa = sum(1.0/np.log(degree_full[w]+1e-9) for w in inter if degree_full.get(w,0)>1)
        pa = deg_u * deg_v
        sc = float(components_full.get(u,-1) == components_full.get(v,-2))
    else:
        cn, aa, pa, sc = 0, 0.0, 0, 0.0
    nfo_sym = nfo_full(u, v) + nfo_full(v, u)
    cos = cosine_sim(u, v)
    return np.array([deg_u, deg_v, cn, aa, pa, sc, nfo_sym, cn*cos], dtype=np.float32)


print("Feature computation")
full_struct_raw = np.stack([pair_features_full(r.u, r.v) for r in train_full.itertuples()])
test_struct_raw = np.stack([pair_features_full(r.u, r.v) for r in test.itertuples()])

scaler_full = StandardScaler().fit(full_struct_raw)
full_struct = scaler_full.transform(full_struct_raw).astype(np.float32)
test_struct2 = scaler_full.transform(test_struct_raw).astype(np.float32)

# PyG graph
all_nodes_full = sorted(G_full.nodes())
n2i_full = {n: i for i, n in enumerate(all_nodes_full)}
x_full_np = np.zeros((len(all_nodes_full), feature_dim), dtype=np.float32)
for node, idx in n2i_full.items():
    if node in node_features_raw: x_full_np[idx] = node_features_raw[node]
x_full_np /= (np.linalg.norm(x_full_np, axis=1, keepdims=True) + 1e-9)
ei_full = torch.tensor([(n2i_full[u], n2i_full[v]) for u,v in G_full.edges()], dtype=torch.long).t().contiguous()
ei_full = torch.cat([ei_full, ei_full.flip(0)], dim=1)
data_full = Data(x=torch.tensor(x_full_np), edge_index=ei_full).to(device)

mask_full = train_full["u"].isin(n2i_full) & train_full["v"].isin(n2i_full)
full_u = torch.tensor([n2i_full[u] for u in train_full["u"][mask_full]], dtype=torch.long, device=device)
full_v = torch.tensor([n2i_full[v] for v in train_full["v"][mask_full]], dtype=torch.long, device=device)
full_s = torch.tensor(full_struct[mask_full.values], dtype=torch.float, device=device)
full_y = torch.tensor(train_full["label"][mask_full].values, dtype=torch.float, device=device)
pw_full = torch.tensor([(full_y==0).sum()/(full_y==1).sum()]).to(device)

# Re training
gnn2 = GNN_model(feature_dim, HIDDEN_SIZE, EMB_SIZE, DROPOUT_GNN).to(device)
predictor2 = LinkPredictor(EMB_SIZE, struct_dim, HIDDEN_SIZE, DROPOUT_MLP).to(device)
opt2 = torch.optim.Adam(list(gnn2.parameters())+list(predictor2.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)

for epoch in range(1, BEST_EPOCH + 1):
    gnn2.train(); predictor2.train()
    opt2.zero_grad()
    z = gnn2(data_full.x, data_full.edge_index, drop_edge_p=DROP_EDGE_P)
    F.binary_cross_entropy_with_logits(predictor2(z[full_u], z[full_v], full_s), full_y, pos_weight=pw_full).backward()
    opt2.step()
    if epoch % 50 == 0:
        print(f"Full retrain epoch {epoch}/{BEST_EPOCH}")

## 8. Prediction on test set

In [ ]:
USE_FULL_RETRAIN = True

if USE_FULL_RETRAIN:
    infer_gnn, infer_pred = gnn2, predictor2
    infer_data, infer_n2i = data_full, n2i_full
    infer_struct = test_struct2
else:
    gnn.load_state_dict({k: v.to(device) for k, v in best_state["gnn"].items()})
    predictor.load_state_dict({k: v.to(device) for k, v in best_state["predictor"].items()})
    infer_gnn, infer_pred = gnn, predictor
    infer_data, infer_n2i = data, node_to_idx
    infer_struct = test_struct

infer_gnn.eval(); infer_pred.eval()
with torch.no_grad():
    z_infer = infer_gnn(infer_data.x, infer_data.edge_index, drop_edge_p=0.0)

infer_s = torch.tensor(infer_struct, dtype=torch.float, device=device)
u_list = [infer_n2i.get(int(r.u), -1) for r in test.itertuples()]
v_list = [infer_n2i.get(int(r.v), -1) for r in test.itertuples()]

scores = []
with torch.no_grad():
    for i, (ui, vi) in enumerate(zip(u_list, v_list)):
        if ui == -1 or vi == -1:
            scores.append(0.1)
        else:
            logit = infer_pred(z_infer[ui].unsqueeze(0),z_infer[vi].unsqueeze(0),infer_s[i].unsqueeze(0))
            scores.append(torch.sigmoid(logit).item())

test["score"] = scores
pd.DataFrame({"ID": range(len(scores)), "Predicted": scores}).to_csv("predictions/GNN_SAGE_NFO_CNCOS.csv", index=False)